In [1]:
import os
import platform
import pandas as pd
import numpy as np
import torch
import pytorch_lightning as pl

from pytorch_lightning.callbacks.early_stopping import EarlyStopping
from torch.utils.data import DataLoader
from multiprocessing import cpu_count

from model.dkt import DKTModule
from model.sakt import SAKTModule
from model.dkvmn import DKVMNModule

In [2]:
seed = 42
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
pl.seed_everything(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

Global seed set to 42


In [3]:
SEQ_LEN = 50
BATCH_SIZE = 64
EMBED_DIM = 128
NUM_WORKERS = 0 if platform.system() == 'Windows' else cpu_count()
# dkt, and sakt, and dkvmn...
model = 'dkt'
q_is_s = False
print("os:{}, num-workers:{}".format(platform.system(), NUM_WORKERS))

os:Linux, num-workers:16


In [4]:
dataset_name = 'assist12'
dataset_path = os.path.join(os.getcwd(), 'dataset', dataset_name)

df = pd.read_csv(os.path.join(dataset_path, "assist.csv"), low_memory=False, encoding="ISO-8859-1").sort_values(by = 'user_id')
key = 'problem_id'

key_skill = 'skill_id' if dataset_name == 'assist09' else 'skill'
key_qtype = 'answer_type' if dataset_name == 'assist09' else 'problem_type'

key_q = 'q_idx'
key_s = 's_idx'

question_id_dict = dict(zip(df[key].unique(), range(len(df[key].unique()))))
skill_id_dict = dict(zip(df[key_skill].unique(), range(len(df[key_skill].unique()))))
user_id_dict = dict(zip(df['user_id'].unique(), range(len(df['user_id'].unique()))))


N_QUESTION = len(question_id_dict)
N_SKILL = len(skill_id_dict)

In [5]:
# 我们需要将数据进行预处理，每个学生的学习记录利用group by合并为序列。
KEY = key_s if q_is_s else key_q
NUM_Q_OR_S = N_SKILL if q_is_s else N_QUESTION

def generate_group_by_df(df):
    group = df[['user_id', KEY, 'correct']].groupby(['user_id']).apply(lambda r: (
            r[KEY].values,
            r['correct'].values
            ))
    return group

df_train, df_test = pd.read_csv(os.path.join(dataset_path, "train.csv"), low_memory=False, encoding="ISO-8859-1"), pd.read_csv(os.path.join(dataset_path, "test.csv"), low_memory=False, encoding="ISO-8859-1")
train, val = generate_group_by_df(df_train), generate_group_by_df(df_test)


In [6]:
from data_loader.dktdataset import DKTDataset

train_dataset = DKTDataset(train, NUM_Q_OR_S, SEQ_LEN)
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)

val_dataset = DKTDataset(val, NUM_Q_OR_S, SEQ_LEN)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print("train:{}, test:{}".format(len(train_dataset), len(val_dataset)))


train:23067, test:5767


In [7]:
import warnings
warnings.filterwarnings('ignore')
if model == 'dkt':
    model = DKTModule(n_question=NUM_Q_OR_S)
elif model == 'sakt':
    model = SAKTModule(n_question=NUM_Q_OR_S, max_seq=SEQ_LEN, embed_dim=EMBED_DIM)
elif model == 'dkvmn':
    model = DKVMNModule(n_question=NUM_Q_OR_S)

print("num of question:{}, num of skill:{}".format(N_QUESTION, N_SKILL))
print("question is skill：{}, num_q_or_s:{}".format(q_is_s, NUM_Q_OR_S))
print("model:{}".format(model))

num of question:50988, num of skill:198
question is skill：False, num_q_or_s:50988
model:DKTModule(
  (loss): BCEWithLogitsLoss()
  (dkt): DKT(
    (embedding): Embedding(101977, 128)
    (lstm): LSTM(128, 256, num_layers=2, batch_first=True, dropout=0.2)
    (pred): Linear(in_features=256, out_features=50988, bias=True)
  )
)


In [8]:
checkpoint_callback = pl.callbacks.ModelCheckpoint(save_top_k=1, verbose=True, monitor='v_auc', mode='max')

# sakt.train_dataloader
trainer = pl.Trainer(
    gpus=1, 
    max_epochs=200, 
    auto_lr_find=True, 
    callbacks=[checkpoint_callback, EarlyStopping(monitor="v_auc", mode="max", patience=6)]
)

trainer.fit(model=model, train_dataloaders=train_dataloader,val_dataloaders=val_dataloader)

GPU available: True, used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name | Type              | Params
-------------------------------------------
0 | loss | BCEWithLogitsLoss | 0     
1 | dkt  | DKT               | 27.1 M
-------------------------------------------
27.1 M    Trainable params
0         Non-trainable params
27.1 M    Total params
108.314   Total estimated model params size (MB)


Sanity Checking: 0it [00:00, ?it/s]

Training: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Epoch 0, global step 361: 'v_auc' reached 0.68580 (best 0.68580), saving model to '/home/czy/KT/BRIKT/lightning_logs/version_43/checkpoints/epoch=0-step=361.ckpt' as top 1


...

Epoch 6, global step 2527: 'v_auc' was not in top 1


Validation: 0it [00:00, ?it/s]

Epoch 7, global step 2888: 'v_auc' was not in top 1


Validation: 0it [00:00, ?it/s]

Epoch 8, global step 3249: 'v_auc' was not in top 1
